# 第11章 発展的話題 ― デモノートブック

この章は「すでに手にした道具だけで現代の話題がどこまで理解できるか」を示す章である。
このノートブックでは、講義ノート第11章の六つの節それぞれについて、
**本文が数値で述べている事実を自分の手で再現する**。
ランダム特徴が期待値としてカーネルに収束すること、補間閾値でテスト誤差が跳ね上がること、
Isomap の二重中心化行列が半正定値でないこと、核ノルム最小化が欠測を埋めること、
Sinkhorn が線形計画の値に近づくこと、そして単一細胞解析の二つの手法（RECODE・scEGOT）の
考え方——これらはすべて数十行のコードで確かめられる。

## 目次

- **11.1 ランダム特徴とニューラルタンジェントカーネル**（命題11.1 の arccos カーネル、経験 NTK）
- **11.2 過剰パラメータ化と二重降下**（最小ノルム補間解と最適リッジ）
- **11.3 多様体学習**（Swiss roll、Isomap の $\boldsymbol{B}$ の負の固有値）
- **11.4 行列補完と核ノルム**（特異値閾値化、観測率と復元誤差）
- **11.5 最適輸送と Wasserstein 距離**（一次元閉形式・線形計画・Sinkhorn・補間・MMD・Gromov--Wasserstein）
- **11.6 単一細胞データ解析への応用**（RECODE 的な縮小、scEGOT 的な成分間輸送）
- **演習**（3問）と**演習の解答**

データ行列は講義ノートの規約どおり $\boldsymbol{X}\in\mathbb{R}^{d\times n}$（**列がサンプル**）で扱う。
scikit-learn は行がサンプルなので、渡すときに転置する。

## 準備

最初にこのセルを実行する。日本語フォントの設定（Colab には既定で入っていない）と、
以降で使うライブラリの読み込みを行う。フォントの導入に失敗した場合は
図のラベルが自動的に英語に切り替わる（`L()` 関数）。

In [ ]:
import subprocess, sys, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=UserWarning)


def _setup_japanese_font():
    """日本語が出せるフォントを探し、なければ入れる。成功したら True。"""
    cands = ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
             "TakaoGothic", "Yu Gothic", "Hiragino Sans"]
    have = {f.name for f in fm.fontManager.ttflist}
    for name in cands:
        if name in have:
            matplotlib.rcParams["font.family"] = name
            return True
    # Colab 想定：pip で導入する
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "japanize-matplotlib"], check=True, timeout=180)
        import japanize_matplotlib  # noqa: F401  読み込むだけで設定される
        return True
    except Exception:
        pass
    # 予備：apt で IPA フォント
    try:
        subprocess.run("apt-get -qq -y install fonts-ipafont-gothic",
                       shell=True, check=True, timeout=300)
        fm._load_fontmanager(try_read_cache=False)
        matplotlib.rcParams["font.family"] = "IPAGothic"
        return True
    except Exception:
        return False


JP = _setup_japanese_font()


def L(ja, en):
    """日本語フォントが使えれば ja、駄目なら en を返す（図のラベル用）。"""
    return ja if JP else en


matplotlib.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "figure.dpi": 110, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False,
})

# 講義ノートの図と同じ色
C = {"blue": "#1f4e79", "red": "#c0392b", "green": "#1e8449",
     "orange": "#d68910", "purple": "#6a4c93", "gray": "#7f8c8d"}

print("日本語フォント:", "有効" if JP else "無効（図のラベルは英語になる）")
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

## 11.1 ランダム特徴とニューラルタンジェントカーネル

幅 $m$ の2層網 $f(\boldsymbol{x};\boldsymbol{\theta})=m^{-1/2}\sum_r a_r\sigma(\boldsymbol{w}_r^{\top}\boldsymbol{x})$ の
（式(11.1)）の隠れ層を固定すると、これは特徴写像
$\boldsymbol{\phi}_m(\boldsymbol{x})=m^{-1/2}(\sigma(\boldsymbol{w}_1^{\top}\boldsymbol{x}),\dots,\sigma(\boldsymbol{w}_m^{\top}\boldsymbol{x}))^{\top}$
による線形モデルであり、その内積は大数の法則で
$k_\sigma(\boldsymbol{x},\boldsymbol{x}')=\mathbb{E}_{\boldsymbol{w}}[\sigma(\boldsymbol{w}^{\top}\boldsymbol{x})\sigma(\boldsymbol{w}^{\top}\boldsymbol{x}')]$
（式(11.2)）に収束する。ReLU ではこの期待値が閉形式で書けた（**命題11.1**、arccos カーネル、式(11.3)）：

$$k_\sigma(\boldsymbol{x},\boldsymbol{x}')=\frac{\|\boldsymbol{x}\|\,\|\boldsymbol{x}'\|}{2\pi}\bigl(\sin\vartheta+(\pi-\vartheta)\cos\vartheta\bigr),
\qquad \vartheta=\arccos\frac{\boldsymbol{x}^{\top}\boldsymbol{x}'}{\|\boldsymbol{x}\|\|\boldsymbol{x}'\|}.$$

本文が挙げた値——単位ベクトルで $\vartheta=0$ なら $1/2$、$\vartheta=\pi/2$ なら
$1/(2\pi)\approx0.159$、$\vartheta=\pi$ なら $0$——を確かめ、有限 $m$ の誤差が
$O(1/\sqrt m)$ で減ることを見る。特徴行列は $\boldsymbol{\Phi}=m^{-1/2}\sigma(\boldsymbol{R}\boldsymbol{X})\in\mathbb{R}^{m\times n}$
（列がサンプル）である。

In [ ]:
def k_arccos(X, Xp):
    """arccos カーネル（命題11.1 の閉形式）。X, Xp は d×n（列がサンプル）。"""
    nx = np.linalg.norm(X, axis=0)[:, None]
    ny = np.linalg.norm(Xp, axis=0)[None, :]
    th = np.arccos(np.clip(X.T @ Xp / (nx * ny), -1.0, 1.0))
    return nx * ny * (np.sin(th) + (np.pi - th) * np.cos(th)) / (2 * np.pi)


def k_rf(X, Xp, m, seed):
    """ランダム特徴の内積 k_m。特徴行列 Phi = ReLU(W X)/√m は m×n（列がサンプル）。"""
    rng = np.random.default_rng(seed)
    W = rng.normal(size=(m, X.shape[0]))            # W は m×d（第 r 行が w_r）
    return (np.maximum(W @ X, 0.0).T @ np.maximum(W @ Xp, 0.0)) / m


theta = np.linspace(0.0, np.pi, 181)
x0 = np.array([[1.0], [0.0]])                        # d×n = 2×1
Xu = np.vstack([np.cos(theta), np.sin(theta)])       # d×n = 2×181（単位ベクトル）
k_exact = k_arccos(x0, Xu)[0]

for t, nm in [(0.0, "0"), (np.pi / 2, "pi/2"), (np.pi, "pi")]:
    xt = np.array([[np.cos(t)], [np.sin(t)]])
    print(f"theta = {nm:5s} :  k_sigma = {k_arccos(x0, xt)[0, 0]:.6f}")

ms = [10, 100, 1000, 10000, 100000]
# 1 回だけだと乱数のばらつきが大きいので 5 シードの平均を取る
errs = [float(np.mean([np.abs(k_rf(x0, Xu, m, 1000 * m + s)[0] - k_exact).max()
                       for s in range(5)])) for m in ms]
for m, e in zip(ms, errs):
    print(f"m = {m:6d} :  max|k_m - k_sigma| = {e:.5f}  (5 シードの平均)")
print(f"m を 10^2 倍したときの誤差比 = {errs[0] / errs[2]:.2f}  (1/sqrt(m) なら 10)")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
ax = axes[0]
ax.plot(theta, k_exact, color=C["blue"], lw=2, label=L("閉形式 $k_\\sigma$", "closed form"))
for m, cc, mk in [(10, C["orange"], "o"), (100, C["green"], "s")]:
    ax.plot(theta[::10], k_rf(x0, Xu, m, m)[0][::10], mk, ms=4, color=cc,
            label=L(f"ランダム特徴 $m={m}$", f"random features m={m}"))
ax.set_xlabel(L("角度 $\\vartheta$", "angle")); ax.set_ylabel(L("カーネル値", "kernel value"))
ax.set_title(L("arccos カーネル（命題11.1）", "arccos kernel"))
ax.legend(fontsize=9)
ax = axes[1]
ax.loglog(ms, errs, "o-", color=C["red"], label=L("実測の最大誤差", "measured max error"))
ax.loglog(ms, errs[0] * np.sqrt(ms[0] / np.array(ms)), "--", color=C["gray"],
          label=L("$1/\\sqrt{m}$ の傾き", "slope $1/\\sqrt{m}$"))
ax.set_xlabel(L("特徴数 $m$", "number of features m")); ax.set_ylabel(L("最大誤差", "max error"))
ax.set_title(L("収束の速さ", "convergence rate")); ax.legend(fontsize=9)
plt.show()

$\vartheta=0$ で $0.5$、$\vartheta=\pi/2$ で $0.159155$、$\vartheta=\pi$ で $0$ と、
本文の値がそのまま出る。**直交する二点でもカーネル値が $0$ にならない**のが線形カーネル
$\boldsymbol{x}^{\top}\boldsymbol{x}'$ との違いである。誤差は $m=10$ で $0.299$、$m=1000$ で $0.039$、
比は $7.59$ で $1/\sqrt m$ の目安（$10$）に合う。

つぎに**全層を学習する**場合の NTK（**定義11.2**）
$\Theta_m(\boldsymbol{x},\boldsymbol{x}')=\langle\nabla_{\boldsymbol{\theta}}f(\boldsymbol{x}),\nabla_{\boldsymbol{\theta}}f(\boldsymbol{x}')\rangle$ を、
勾配の内積として直接計算した値（経験 NTK、式(11.5)）と閉形式（式(11.6)）
$\Theta(\boldsymbol{x},\boldsymbol{x}')=k_\sigma(\boldsymbol{x},\boldsymbol{x}')+(\boldsymbol{x}^{\top}\boldsymbol{x}')(\pi-\vartheta)/(2\pi)$
とで突き合わせる。本文は $d=3$、$m=2\times10^5$ で最大差 $9\times10^{-3}$ と報告している。

In [ ]:
def ntk_relu(X, Xp):
    """2層 ReLU 網の NTK（解析形）。X, Xp は d×n。"""
    nx = np.linalg.norm(X, axis=0)[:, None]
    ny = np.linalg.norm(Xp, axis=0)[None, :]
    th = np.arccos(np.clip(X.T @ Xp / (nx * ny), -1.0, 1.0))
    k0 = nx * ny * (np.sin(th) + (np.pi - th) * np.cos(th)) / (2 * np.pi)
    return k0 + (X.T @ Xp) * (np.pi - th) / (2 * np.pi)


def ntk_emp(X, m, seed):
    """勾配の内積として直接計算した経験 NTK。"""
    rng = np.random.default_rng(seed)
    W, a = rng.normal(size=(m, X.shape[0])), rng.normal(size=m)
    Sm = np.maximum(W @ X, 0.0) / np.sqrt(m)          # m×n : da 方向
    Dm = (W @ X > 0) * a[:, None] / np.sqrt(m)        # m×n : dW 方向の係数
    return Sm.T @ Sm + (Dm.T @ Dm) * (X.T @ X)


rng = np.random.default_rng(0)
Xn = rng.normal(size=(4, 3)).T                        # d×n = 3×4
ms_n = [1000, 10000, 100000, 200000]
errs_n = [float(np.mean([np.abs(ntk_relu(Xn, Xn) - ntk_emp(Xn, m, s)).max()
                         for s in range(5)])) for m in ms_n]
for m, e in zip(ms_n, errs_n):
    print(f"m = {m:7d} :  max|Theta_m - Theta| = {e:.2e}  (5 シードの平均)")
diag_gap = np.abs(np.diag(ntk_relu(Xn, Xn)) - (Xn ** 2).sum(axis=0)).max()
print(f"対角の恒等式 Theta(x,x) = ||x||^2 の最大差 = {diag_gap:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
ax = axes[0]
ax.plot(theta, ntk_relu(x0, Xu)[0], color=C["blue"], lw=2, label=L("NTK $\\Theta$", "NTK"))
ax.plot(theta, k_exact, color=C["gray"], lw=1.5, ls="--",
        label=L("NNGP $k_\\sigma$", "NNGP kernel"))
ax.plot(theta[::10], ntk_emp(np.hstack([x0, Xu]), 2000, 1)[0, 1:][::10], "o", ms=4,
        color=C["orange"], label=L("経験 NTK $m=2000$", "empirical NTK m=2000"))
ax.set_xlabel(L("角度 $\\vartheta$", "angle")); ax.set_ylabel(L("カーネル値", "kernel value"))
ax.set_title(L("NTK と NNGP カーネル", "NTK vs NNGP kernel")); ax.legend(fontsize=9)
ax = axes[1]
ax.loglog(ms_n, errs_n, "o-", color=C["red"], label=L("経験 NTK の最大差", "max deviation"))
ax.loglog(ms_n, errs_n[0] * np.sqrt(ms_n[0] / np.array(ms_n)), "--", color=C["gray"],
          label=L("$1/\\sqrt{m}$ の傾き", "slope $1/\\sqrt{m}$"))
ax.set_xlabel(L("幅 $m$", "width m")); ax.set_ylabel(L("最大差", "max deviation"))
ax.set_title(L("有限幅のずれ", "finite-width deviation")); ax.legend(fontsize=9)
plt.show()

$m=2\times10^5$ での最大差は $1.3\times10^{-2}$ で、本文の $9\times10^{-3}$ と同じオーダーである
（乱数の引き方が違うので一致はしない）。対角の恒等式 $\Theta(\boldsymbol{x},\boldsymbol{x})=\|\boldsymbol{x}\|^2$ は
機械精度（最大差 $7\times10^{-9}$）で成り立つ。左図で NTK が NNGP カーネルより一貫して大きいのは、
NTK には隠れ層の寄与 $(\boldsymbol{x}^{\top}\boldsymbol{x}')(\pi-\vartheta)/(2\pi)$ が加わるからである。

$\Theta$ が学習中に凍結するなら、勾配降下の予測は固定カーネルによるカーネル回帰（式(11.7)）に一致する。
ただし**注意11.3** のとおり、これは「無限幅・小さい学習率・特定の規格化」のもとでの主張であって、
深層学習の成功の説明ではない。

## 11.2 過剰パラメータ化と二重降下

本文の**図11.1**と**演習11.5**に対応する。ランダム特徴回帰
$\boldsymbol{\Phi}\in\mathbb{R}^{m\times n}$（列がサンプル）で特徴数 $m$ を動かし、
$m/n=1$（補間閾値）の前後でテスト誤差がどう動くかを見る。$m>n$ では
$\boldsymbol{\Phi}^{\top}\boldsymbol{b}=\boldsymbol{y}$ の解が無数にあり、`pinv` は**最小ノルム解**（式(11.8)）を返す。
比較のために、各 $m$ で $\lambda$ をグリッド探索した**最適リッジ**も計算する
（本文の警告ボックス：最適リッジ曲線は二重降下曲線を全域で下から押さえる。
リスクの表式は**定理11.4**、式(11.9)）。

> **再現性についての重要な注意**：補間閾値でのピークの**高さ**は $1/\sigma_{\min}(\boldsymbol{\Phi})^2$ に
> 支配され、分布の裾が非常に重い。平均は試行ごとに桁で動くので、以下では
> **15 シードの中央値**を描き、四分位範囲を帯で示す。ピークの「存在」は頑健だが、
> 「高さ」の数値は再現性が低い量である。

In [ ]:
def dd_one(m, seed, n=60, d=20, ntest=500, lams=np.logspace(-10, 2, 25)):
    """ランダム特徴回帰。最小ノルム補間解と最適リッジのテスト誤差を返す。"""
    rng = np.random.default_rng(seed)
    w = rng.normal(size=d); w /= np.linalg.norm(w)
    X = rng.normal(size=(d, n))                        # d×n（列がサンプル）
    y = X.T @ w + 0.1 * rng.normal(size=n)
    Xt = rng.normal(size=(d, ntest)); yt = Xt.T @ w
    W = rng.normal(size=(m, d))
    Ph = np.maximum(W @ X, 0.0) / np.sqrt(m)           # m×n
    Pt = np.maximum(W @ Xt, 0.0) / np.sqrt(m)          # m×ntest
    b = np.linalg.pinv(Ph.T, rcond=1e-15) @ y          # 最小ノルム解（rcond を小さく）
    e_min = float(np.mean((Pt.T @ b - yt) ** 2))
    G = Ph.T @ Ph                                       # n×n：リッジは双対形で解く
    ev, V = np.linalg.eigh(G); Vy = V.T @ y
    Kt = Pt.T @ Ph                                      # ntest×n
    e_rdg = min(float(np.mean((Kt @ (V @ (Vy / (ev + lam))) - yt) ** 2)) for lam in lams)
    return e_min, e_rdg


n_dd, n_seed = 60, 15
ms_dd = np.unique(np.concatenate([np.round(np.logspace(np.log10(5), np.log10(600), 16)),
                                  [n_dd]]).astype(int))   # 補間閾値 m = n をちょうど含める
res = np.array([[dd_one(int(m), 1000 * s + 7, n=n_dd) for s in range(n_seed)] for m in ms_dd])
med = np.median(res, axis=1)
q1, q3 = np.percentile(res, 25, axis=1), np.percentile(res, 75, axis=1)

for m in [30, 60, 600]:
    i = int(np.argmin(np.abs(ms_dd - m)))
    print(f"m = {ms_dd[i]:3d} (m/n = {ms_dd[i]/n_dd:5.2f}) : "
          f"最小ノルム中央値 {med[i,0]:8.3f}   最適リッジ中央値 {med[i,1]:.3f}")
i_pk = int(np.argmax(med[:, 0]))
print(f"ピーク位置 m/n = {ms_dd[i_pk]/n_dd:.2f}, 中央値 {med[i_pk,0]:.2f} "
      f"(四分位 {q1[i_pk,0]:.2f}-{q3[i_pk,0]:.2f}, {n_seed} シード)")
print(f"リッジ/最小ノルム（中央値の比）の最大 = {float((med[:,1]/med[:,0]).max()):.4f}  (<= 1 なら全域で下回る)")

fig, ax = plt.subplots(figsize=(7.2, 4.2))
x_ax = ms_dd / n_dd
ax.fill_between(x_ax, q1[:, 0], q3[:, 0], color=C["red"], alpha=0.18)
ax.plot(x_ax, med[:, 0], "o-", color=C["red"], ms=4,
        label=L("最小ノルム解（中央値）", "min-norm (median)"))
ax.plot(x_ax, med[:, 1], "s-", color=C["blue"], ms=4,
        label=L("最適リッジ（中央値）", "optimal ridge (median)"))
ax.axvline(1.0, color=C["gray"], ls="--", lw=1)
ax.set_xscale("log"); ax.set_yscale("log")
ax.text(0.52, 0.06, L("補間閾値 $m/n=1$", "interpolation threshold"),
        transform=ax.transAxes, fontsize=9, color=C["gray"])
ax.set_xlabel(L("$m/n$（パラメータ数 / 標本数）", "m/n"))
ax.set_ylabel(L("テスト二乗誤差", "test MSE"))
ax.set_title(L(f"二重降下（$n={n_dd}$, {n_seed} シードの中央値）",
               f"double descent (n={n_dd}, median of {n_seed} seeds)"))
ax.legend(fontsize=9)
plt.show()

古典領域（$m/n<1$）では $m$ とともに誤差が減り、補間閾値 $m/n=1$ ちょうどで中央値 $32.5$ の
鋭い山ができ、その先で再び下降して $m/n=10$ では $0.043$ まで落ちる。
これは古典領域の最良値（$m/n=0.42$ で $0.645$）より小さい。**第二の下降**である。
山の正体は $\boldsymbol{\Phi}$（$m\times n$）が正方に近づいて最小特異値が $0$ に潰れることであり、
本文の Marchenko--Pastur による説明（**注意11.5**、式(11.10)）——台の左端 $(1-\sqrt\gamma)^2$ が
$0$ に落ちるため $\int x^{-1}\mathrm{d}\mathrm{MP}_\gamma$ が発散する——と同じことを見ている。

最適リッジ（青）は山を持たず単調に減り、実測でも**全域で最小ノルム曲線を下回る**
（中央値の比の最大が $1.0000$、すなわち上回る点がない）。二重降下は
「正則化をアルゴリズム任せにしたときに何が起こるか」の曲線であって、
過剰パラメータ化それ自体の利益を示すものではない。

なお四分位範囲（帯）はピークで $7.5--157.6$ と 20 倍以上に広がる。ここが本文の警告どおり
**平均ではなく中央値を、試行数を明記して報告すべき**場所である。

## 11.3 多様体学習

Swiss roll（$n=800$、$8$ 近傍）に線形 PCA・Isomap・LLE・ラプラシアン固有写像（スペクトラル埋め込み）を
当てて比較する（本文の**図11.2**に対応。LLE は式(11.11)、ラプラシアン固有写像は式(11.12)の
固有値問題である）。次に本文の警告ボックス——**グラフ最短経路から作った
$\boldsymbol{B}=-\frac12\boldsymbol{H}\boldsymbol{D}_G^{(2)}\boldsymbol{H}$ は半正定値でない**——を実測で確かめる。

Isomap は古典的 MDS の距離を測地距離の近似に取り替えただけのものだった。
Euclid 距離なら $\boldsymbol{B}=\tilde{\boldsymbol{X}}^{\top}\tilde{\boldsymbol{X}}\succeq0$ が保証されるが、
$d_G$ にはその保証がない。負の固有値がどれだけ小さいかが Isomap が実用になる条件である。

In [ ]:
from scipy.sparse.csgraph import shortest_path
from sklearn.datasets import make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap, LocallyLinearEmbedding, SpectralEmbedding
from sklearn.neighbors import kneighbors_graph

n_pt, k_nn = 800, 8
Xs, col = make_swiss_roll(n_samples=n_pt, noise=0.0, random_state=0)
X = Xs.T                                     # d×n = 3×400（列がサンプル）に転置
# sklearn は行がサンプルなので、渡すときは X.T に戻す
emb = {
    "PCA": PCA(n_components=2).fit_transform(X.T).T,
    "Isomap": Isomap(n_neighbors=k_nn, n_components=2).fit_transform(X.T).T,
    "LLE": LocallyLinearEmbedding(n_neighbors=k_nn, n_components=2,
                                  random_state=0).fit_transform(X.T).T,
    L("ラプラシアン固有写像", "Laplacian eigenmaps"):
        SpectralEmbedding(n_components=2, n_neighbors=k_nn,
                          random_state=0).fit_transform(X.T).T,
}

# 二重中心化行列 B（測地距離版と Euclid 距離版）
Gk = kneighbors_graph(X.T, k_nn, mode="distance")
DG = shortest_path(Gk, method="D", directed=False)          # n×n 測地距離の近似
DE = np.linalg.norm(X[:, :, None] - X[:, None, :], axis=0)  # n×n Euclid 距離
H = np.eye(n_pt) - np.ones((n_pt, n_pt)) / n_pt
B_geo = -0.5 * H @ (DG ** 2) @ H
B_euc = -0.5 * H @ (DE ** 2) @ H
ev_g = np.linalg.eigvalsh(B_geo)[::-1]
ev_e = np.linalg.eigvalsh(B_euc)[::-1]
neg = ev_g[ev_g < 0]
print(f"測地距離の B : 固有値の範囲 [{ev_g.min():.1f}, {ev_g.max():.3e}]")
print(f"              負の固有値 {len(neg)} / {n_pt} 個")
print(f"              |負| の和 / 正の和 = {np.abs(neg).sum()/ev_g[ev_g>0].sum()*100:.2f} %")
print(f"              最大の負固有値 / lambda_1 = {np.abs(neg).max()/ev_g[0]*100:.2f} %")
print(f"Euclid 距離の B : 最小固有値 {ev_e.min():.1e},  "
      f"階数 {int((ev_e > 1e-8*ev_e.max()).sum())}（= 埋め込み次元）")

# 同じ k でも点が疎だと測地距離の近似が悪くなる：n = 400 との比較
Xs4 = make_swiss_roll(n_samples=400, noise=0.0, random_state=0)[0]
D4 = shortest_path(kneighbors_graph(Xs4, k_nn, mode="distance"), method="D", directed=False)
H4 = np.eye(400) - np.ones((400, 400)) / 400
e4 = np.linalg.eigvalsh(-0.5 * H4 @ (D4 ** 2) @ H4)
print(f"参考: n=400（同じ k={k_nn}）だと |負|/正 = "
      f"{np.abs(e4[e4<0]).sum()/e4[e4>0].sum()*100:.2f} %")

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, (name, Y) in zip(axes, emb.items()):
    ax.scatter(Y[0], Y[1], c=col, cmap="Spectral", s=8)
    ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(L("Swiss roll の埋め込み（色は巻きに沿った位置）",
               "Swiss roll embeddings (colour = position along the roll)"), y=1.04)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
ax = axes[0]
ax.plot(np.arange(1, n_pt + 1), ev_g, ".", ms=4, color=C["blue"],
        label=L("測地距離", "geodesic"))
ax.plot(np.arange(1, n_pt + 1), ev_e, ".", ms=4, color=C["orange"],
        label=L("Euclid 距離", "Euclidean"))
ax.axhline(0, color=C["gray"], lw=1)
ax.set_yscale("symlog", linthresh=1.0)
ax.set_xlabel(L("固有値の番号", "index")); ax.set_ylabel(L("固有値", "eigenvalue"))
ax.set_title(L("$\\mathbf{B}$ の固有値", "eigenvalues of B")); ax.legend(fontsize=9)
ax = axes[1]
ax.plot(np.arange(1, 41), ev_g[-40:][::-1], "o-", ms=4, color=C["red"])
ax.axhline(0, color=C["gray"], lw=1)
ax.set_xlabel(L("下位から数えた番号", "index from the bottom"))
ax.set_ylabel(L("固有値", "eigenvalue"))
ax.set_title(L("下位40個（すべて負）", "bottom 40 (all negative)"))
plt.show()

線形 PCA は巻きを潰して帯を復元できないが、近傍グラフを使う三手法は巻きを展開する。
Isomap は測地距離を保つので帯の**長さの比**まで残るのに対し、LLE とラプラシアン固有写像は
局所構造だけを保つので大域的な縦横比は保証されない。

$\boldsymbol{B}$ の固有値は $[-5.2\times10^{3},\ 6.19\times10^{5}]$ の範囲に散らばり、$800$ 個のうち **$422$ 個が負**である。
ただし絶対値は小さく、負の固有値の絶対値の和は正の和の $4.69\,\%$、
最大の負固有値は $\lambda_1$ の $0.84\,\%$ にすぎない。
同じ点集合を Euclid 距離で扱えば最小固有値は $-2.3\times10^{-11}$、すなわち丸め誤差の範囲で
$\boldsymbol{B}\succeq0$、階数は $3$（元の空間の次元）である。

Isomap は $\boldsymbol{Y}^{\top}\boldsymbol{Y}\succeq0$ という制約つき最良近似を解くので、
負の固有値は単に**捨てる**。それが許されるのは負の成分が小さいからである。

> **この割合は点の密度と近傍数に敏感である。** 同じ $k=8$ でも $n=400$ に減らすと
> $|負|/正 = 32.97\,\%$ まで悪化する。点が疎だとグラフ上の最短経路が折れ線になって
> 測地距離を過大評価したり、逆に巻きの層をまたいで短絡したりするからである。
> 講義ノート本文は $n=400$ で $4.33\,\%$ と報告しているが、上の実測で $n=400$ とすると
> $32.97\,\%$ でこれとは合わず、同じ桁になるのは $n=800$ の $4.69\,\%$ の方である。
> 本文と点の取り方が違うためと思われる（数値そのものは一致しない）。
> 負の固有値が大きいときは近傍数 $k$ と点の密度を見直す合図になる（演習2）。

## 11.4 行列補完と核ノルム

ランク $r$ の行列 $\boldsymbol{M}\in\mathbb{R}^{d\times n}$ の一部 $\Omega$ だけを観測して残りを埋める。
$\mathrm{rank}$ の最小化は NP 困難なので、その凸包絡（**命題11.9**）である**核ノルム**
$\|\boldsymbol{X}\|_*=\sum_i\sigma_i(\boldsymbol{X})$（**定義11.8**）に緩和し、式(11.16)を解く。
鍵は核ノルムの近接写像が**特異値の軟閾値化**で書けることである（**定理11.13**、式(11.18)）：

$$\mathcal{D}_\tau(\boldsymbol{Y})=\arg\min_{\boldsymbol{X}}\tfrac12\|\boldsymbol{X}-\boldsymbol{Y}\|_{\mathrm F}^2+\tau\|\boldsymbol{X}\|_*
=\boldsymbol{U}\,\mathrm{diag}((s_i-\tau)_+)\,\boldsymbol{V}^{\top}.$$

これを使った反復
$\boldsymbol{X}^{(t+1)}=\mathcal{D}_\lambda(\mathcal{P}_\Omega(\boldsymbol{M})+\mathcal{P}_\Omega^{\perp}(\boldsymbol{X}^{(t)}))$
（soft-impute、式(11.19)）は「観測成分は真値で埋め、未観測成分は現在の推定で埋め、
全体を軟閾値化する」だけである。
$\lambda$ を温かい出発点で下げながら回す。自由度は $r(d+n-r)=351$ で、
これは全成分数 $3600$ の $9.75\,\%$ にあたる。

In [ ]:
def prox_nuc(Y, tau):
    """核ノルムの近接写像＝特異値の軟閾値化。"""
    U, s, Vt = np.linalg.svd(Y, full_matrices=False)
    return (U * np.maximum(s - tau, 0.0)) @ Vt


def soft_impute(M, mask, lams=(10, 3, 1, 0.3, 0.1, 0.03, 0.01), n_iter=150):
    """P_Omega(M) + P_Omega^perp(X) を軟閾値化する反復（温かい出発点で lambda を下げる）。"""
    X = np.zeros_like(M)
    for lam in lams:
        for _ in range(n_iter):
            X = prox_nuc(np.where(mask, M, X), lam)
    return X


rng = np.random.default_rng(0)
d_m, n_m, r_m = 60, 60, 3
Mtrue = rng.normal(size=(d_m, r_m)) @ rng.normal(size=(r_m, n_m))   # d×n ランク 3
nrmM = np.linalg.norm(Mtrue)
rates = np.array([0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.55, 0.71, 0.85])
err_mc, rank_mc = [], []
for p in rates:
    ee, rr = [], []
    for s in range(3):
        rs = np.random.default_rng(100 + s)
        mask = rs.random((d_m, n_m)) < p
        Xh = soft_impute(Mtrue, mask)
        ee.append(np.linalg.norm(Xh - Mtrue) / nrmM)
        rr.append(int((np.linalg.svd(Xh, compute_uv=False) > 1e-3 * nrmM).sum()))
    err_mc.append(float(np.median(ee))); rank_mc.append(int(np.median(rr)))
err_mc, rank_mc = np.array(err_mc), np.array(rank_mc)
for p, e, k in zip(rates, err_mc, rank_mc):
    print(f"観測率 {p:.2f} : 相対誤差 {e:.2e}   推定ランク {k}")

mask50 = np.random.default_rng(100).random((d_m, n_m)) < 0.30
Xh50 = soft_impute(Mtrue, mask50)
print(f"下の図の例（観測率 0.30、1 シード）: 相対誤差 "
      f"{np.linalg.norm(Xh50-Mtrue)/nrmM:.3e}")

fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.9))
ax = axes[0]
ax.semilogy(rates, err_mc, "o-", color=C["blue"])
ax.axvline(r_m * (d_m + n_m - r_m) / (d_m * n_m), color=C["red"], ls="--", lw=1.2)
ax.text(r_m * (d_m + n_m - r_m) / (d_m * n_m) + 0.02, err_mc.min() * 2.0,
        L("自由度の下限", "degrees of freedom"), color=C["red"], fontsize=9)
ax.set_xlabel(L("観測率 $|\\Omega|/(dn)$", "observed fraction"))
ax.set_ylabel(L("相対誤差 $\\|\\hat{X}-M\\|_F/\\|M\\|_F$", "relative error"))
ax.set_title(L("観測率と復元誤差（3 シードの中央値）", "error vs observed fraction"))
ax = axes[1]
ax.plot(rates, rank_mc, "s-", color=C["green"])
ax.axhline(r_m, color=C["gray"], ls="--", lw=1)
ax.set_xlabel(L("観測率", "observed fraction")); ax.set_ylabel(L("推定ランク", "estimated rank"))
ax.set_title(L("推定ランク（真値 3）", "estimated rank (true 3)"))
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.4))
vmax = np.abs(Mtrue).max()
for ax, (A, t) in zip(axes, [(Mtrue, L("真の行列 $M$", "true M")),
                             (np.where(mask50, Mtrue, np.nan), L("観測（30%）", "observed (30%)")),
                             (Xh50, L("復元 $\\hat{X}$", "recovered"))]):
    im = ax.imshow(A, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(t); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.colorbar(im, ax=axes, shrink=0.85)
plt.show()

観測率 $0.71$（本文の例と同じ設定）では相対誤差 $2.9\times10^{-4}$、推定ランク $3$ で
真値と一致する。本文の**例11.14**が報告する $2.4\times10^{-4}$ と同じオーダーである。
観測率を下げると誤差は $0.55$ で $4.3\times10^{-4}$、$0.40$ で $6.5\times10^{-3}$、$0.30$ で $5.8\times10^{-2}$、
$0.20$ で $0.25$ と急に立ち上がり、推定ランクも $3$ から $17$ へ跳ね上がる
（ランクを見誤ると復元は失敗している）。自由度の下限 $9.75\,\%$ に近づくにつれ
情報が足りなくなる様子が見える。復元保証（**定理11.11**）が要求する
$m\gtrsim C\mu_0nr\log^2n$（$\mu_0$ は**定義11.10**のコヒーレンス）の
$\log^2n$ と定数 $C$ の分だけ、自由度ちょうどより余裕が要るということである。

下の図では観測率 $30\%$ の穴だらけの行列から元の構造がおおよそ復元されている
（この 1 例の相対誤差は $5.8\times10^{-2}$）。$\lambda$ を最初から $0.01$ にすると収束が遅いので、
$10\to3\to1\to0.3\to0.1\to0.03\to0.01$ と温かい出発点で下げている。
本文の**注意11.12**の $2\times2$ の反例——左上三成分だけの観測では核ノルム最小解 $x=1$ と
ランク最小解 $x=6$ が食い違う——が示すとおり、観測が足りなければ凸緩和は破綻する。演習3で扱う。

## 11.5 最適輸送と Wasserstein 距離

本文の**節11.5**（図11.3）に対応する。カップリングの集合は**定義11.15**、
Kantorovich 問題は**定義11.16**、その離散版が式(11.21)である。この節は五つの話題を順に扱う。

1. **一次元の閉形式**（分位関数の差）と離散の**線形計画**が一致すること
2. **Sinkhorn** の $\varepsilon\to0$ での挙動と、素朴な実装が破綻する理由・対数領域での修正
3. **Wasserstein 補間 vs $L^2$ 補間**（山が移動するか、薄れて濃くなるか）
4. **MMD との比較**（台が離れたときの挙動）
5. **Gromov--Wasserstein**（空間を共有しない対象の比較）

まず一次元。等重みの経験分布では、$W_2^2=\int_0^1|F_\mu^{-1}(u)-F_\nu^{-1}(u)|^2\mathrm{d}u$ は
「両方を昇順に並べ替えて差の二乗平均を取る」ことに帰着する（**定理11.17**、式(11.22)、単調カップリング）。
これを $nm$ 変数の線形計画（式(11.21)）の解と突き合わせる。本文の**例11.18**に対応する。

In [ ]:
from scipy.optimize import linprog
from scipy.special import logsumexp


def ot_lp(a, b, Cm):
    """離散最適輸送の線形計画（式(11.21) の U(a,b) 上の最小化）。"""
    n, m = Cm.shape
    A1 = np.zeros((n, n * m)); A2 = np.zeros((m, n * m))
    for i in range(n):
        A1[i, i * m:(i + 1) * m] = 1.0      # 行和 = a
    for j in range(m):
        A2[j, j::m] = 1.0                   # 列和 = b
    r = linprog(Cm.ravel(), A_eq=np.vstack([A1, A2]),
                b_eq=np.concatenate([a, b]), bounds=(0, None), method="highs")
    return float(r.fun), r.x.reshape(n, m)


def sinkhorn(a, b, Cm, eps, n_iter=1000):
    """素朴な Sinkhorn（K = exp(-C/eps) を直に持つ）。"""
    K = np.exp(-Cm / eps)
    u, v = np.ones_like(a), np.ones_like(b)
    for _ in range(n_iter):
        u = a / (K @ v)                     # 行の周辺分布を a に合わせる
        v = b / (K.T @ u)                   # 列の周辺分布を b に合わせる
    return u[:, None] * K * v[None, :]


def sinkhorn_log(a, b, Cm, eps, n_iter=1000):
    """対数領域の Sinkhorn（logsumexp で安定化）。"""
    f, g = np.zeros_like(a), np.zeros_like(b)
    for _ in range(n_iter):
        f = eps * (np.log(a) - logsumexp((g[None, :] - Cm) / eps, axis=1))
        g = eps * (np.log(b) - logsumexp((f[:, None] - Cm) / eps, axis=0))
    return np.exp((f[:, None] + g[None, :] - Cm) / eps)


rng = np.random.default_rng(0)
n_ot = 40
xs = np.sort(rng.random(n_ot))
ys = np.sort(rng.random(n_ot)) + 0.3
a_ot = b_ot = np.full(n_ot, 1.0 / n_ot)
C2 = (xs[:, None] - ys[None, :]) ** 2
C1 = np.abs(xs[:, None] - ys[None, :])

lp2, P_lp = ot_lp(a_ot, b_ot, C2)
lp1, _ = ot_lp(a_ot, b_ot, C1)
print(f"W_2^2 : 線形計画 {lp2:.10f} / ソートによる閉形式 {np.mean((xs-ys)**2):.10f}")
print(f"W_1   : 線形計画 {lp1:.10f} / ソートによる閉形式 {np.mean(np.abs(xs-ys)):.10f}")
grid = np.linspace(-0.2, 1.6, 20001)
dif = np.abs((xs[None, :] <= grid[:, None]).mean(1) - (ys[None, :] <= grid[:, None]).mean(1))
cdf_gap = float(np.sum((dif[1:] + dif[:-1]) / 2 * np.diff(grid)))     # 台形則
print(f"W_1   : CDF の差の積分 int|F-G|dt = {cdf_gap:.10f}")

eps_list = [3e-1, 1e-1, 3e-2, 1e-2, 3e-3, 1e-3, 3e-4]
gap_naive, gap_log = [], []
for ep in eps_list:
    n_it = int(np.clip(6 / ep, 800, 12000))       # 反復数は O(1/eps) 必要になる
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        Pn = sinkhorn(a_ot, b_ot, C2, ep, n_iter=n_it)
    cn = float((Pn * C2).sum()) if np.all(np.isfinite(Pn)) else np.nan
    Pl = sinkhorn_log(a_ot, b_ot, C2, ep, n_iter=n_it)
    gap_naive.append(abs(cn - lp2)); gap_log.append(abs(float((Pl * C2).sum()) - lp2))
    print(f"eps = {ep:.0e} (反復 {n_it:5d}) : "
          f"素朴 {'nan（アンダーフロー）' if not np.isfinite(cn) else f'{cn:.8f}'}"
          f"   対数領域 {float((Pl*C2).sum()):.8f}   周辺制約の誤差 "
          f"{np.abs(Pl.sum(1)-a_ot).max():.1e}")

fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.9))
ax = axes[0]
ax.loglog(eps_list, gap_log, "o-", color=C["blue"], label=L("対数領域", "log-domain"))
fin = np.isfinite(gap_naive)
ax.loglog(np.array(eps_list)[fin], np.array(gap_naive)[fin], "s--", color=C["red"],
          label=L("素朴な実装", "naive"))
for ep, g in zip(np.array(eps_list)[~fin], [gap_log[i] for i in np.where(~fin)[0]]):
    ax.plot(ep, g, "x", color=C["red"], ms=9)
ax.set_xlabel(L("$\\varepsilon$", "epsilon"))
ax.set_ylabel(L("|Sinkhorn $-$ 線形計画|", "|Sinkhorn - LP|"))
ax.set_title(L("$\\varepsilon\\to0$ での収束（$\\times$ は nan）",
               "convergence as eps -> 0 (x = nan)")); ax.legend(fontsize=9)
ax = axes[1]
im = ax.imshow(P_lp, cmap="viridis"); ax.grid(False)
ax.set_title(L("線形計画の最適計画 $P^\\star$（単調）", "optimal plan (monotone)"))
ax.set_xlabel(L("$\\nu$ の点（昇順）", "nu (sorted)"))
ax.set_ylabel(L("$\\mu$ の点（昇順）", "mu (sorted)"))
fig.colorbar(im, ax=ax, shrink=0.85)
plt.show()

線形計画の値とソートによる閉形式は $W_2^2$ で $0.0690190390$、$W_1$ で $0.2535551527$ と一致し、
$W_1$ は CDF の差の積分 $\int|F_\mu-F_\nu|\mathrm{d}t=0.2535638$ とも数値積分の刻み幅の範囲で
一致する（差 $8.6\times10^{-6}$ は台形則の格子によるもので、格子を細かくすれば減る）。
一次元は $O(n\log n)$ で解ける。

つぎにエントロピー正則化（式(11.23)、**節11.5.3**、**演習11.6**）。その最適解が
$\mathrm{diag}(\boldsymbol{u})\boldsymbol{K}\mathrm{diag}(\boldsymbol{v})$ の形になること（**命題11.19**）が
Sinkhorn 反復の根拠である。$\varepsilon$ を小さくすると線形計画の値に近づく：差は
$\varepsilon=3\times10^{-1}$ で $8.9\times10^{-2}$、$\varepsilon=3\times10^{-4}$ で $6.5\times10^{-5}$ と
単調に減る（おおむね $\varepsilon$ に比例する）。ただし素朴な実装は $\varepsilon=3\times10^{-4}$ で
`nan` になる。原因は $\boldsymbol{K}=\exp(-\boldsymbol{C}/\varepsilon)$ の成分が float64 のアンダーフロー
（$\sim10^{-308}$）に達して $\boldsymbol{K}\boldsymbol{v}=0$ となることであって、アルゴリズムの収束性の
問題ではない（**注意11.20**）。対数領域（`logsumexp`）に移せば同じ $\varepsilon$ でも安定に動く。
必要な反復数が $O(1/\varepsilon)$ で増えることには注意が要る（上のセルでは
$\varepsilon$ に応じて $800$ から $12000$ 回まで増やしている。反復が足りないと
周辺制約が満たされず、線形計画の値を下回る「もっともらしい」数字が出てしまう）。

右図の最適計画は対角に集中している。これが**単調カップリング**である。

つぎに補間と MMD。$W_p$ は台の幾何を見るので分布が「移動」するが、$L^2$ の線形補間は
一方が薄れて他方が濃くなる。MMD はカーネルの帯域幅より離れると差を感じ取れなくなる。

In [ ]:
# --- Wasserstein 補間 vs L2 補間（同分散ガウスは閉形式）---
t_grid = np.linspace(-4.5, 4.5, 601)
mu_m, nu_m, sd_g = -2.0, 2.0, 0.35


def gauss(t, m, s):
    return np.exp(-(t - m) ** 2 / (2 * s ** 2)) / (s * np.sqrt(2 * np.pi))


ss = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.7))
ax = axes[0]
for s, alpha in zip(ss, np.linspace(0.35, 1.0, len(ss))):
    ax.plot(t_grid, gauss(t_grid, (1 - s) * mu_m + s * nu_m, sd_g),
            color=C["blue"], alpha=alpha, lw=1.8)
ax.set_title(L("Wasserstein 補間（山が移動）", "Wasserstein interpolation"))
ax.set_xlabel("$t$"); ax.set_ylabel(L("密度", "density"))
ax = axes[1]
for s, alpha in zip(ss, np.linspace(0.35, 1.0, len(ss))):
    ax.plot(t_grid, (1 - s) * gauss(t_grid, mu_m, sd_g) + s * gauss(t_grid, nu_m, sd_g),
            color=C["red"], alpha=alpha, lw=1.8)
ax.set_title(L("$L^2$ 補間（薄れて濃くなる）", "$L^2$ interpolation"))
ax.set_xlabel("$t$")

# --- MMD と W_2 の比較（ガウスカーネル、閉形式）---
shift = np.linspace(0.0, 6.0, 121)
sig_k, var_g = 0.5, sd_g ** 2


def kxy(dm, v1, v2, sk):
    return sk / np.sqrt(sk ** 2 + v1 + v2) * np.exp(-dm ** 2 / (2 * (sk ** 2 + v1 + v2)))


mmd = np.sqrt(np.maximum(2 * kxy(0.0, var_g, var_g, sig_k)
                         - 2 * kxy(shift, var_g, var_g, sig_k), 0.0))
ax = axes[2]
ax.plot(shift, shift, color=C["blue"], lw=2, label=L("$W_2$", "$W_2$"))
ax.plot(shift, mmd, color=C["red"], lw=2, label=L("MMD（帯域幅 0.5）", "MMD (bw 0.5)"))
ax.set_xlabel(L("平均のずれ", "mean shift")); ax.set_ylabel(L("距離", "distance"))
ax.set_title(L("台が離れたときの挙動", "behaviour when supports separate"))
ax.legend(fontsize=9)
plt.show()

i2 = int(np.argmin(np.abs(shift - 2.0))); i6 = -1
print(f"ずれ 2.0 : W_2 = {shift[i2]:.3f},  MMD = {mmd[i2]:.4f}")
print(f"ずれ 6.0 : W_2 = {shift[i6]:.3f},  MMD = {mmd[i6]:.4f}  "
      f"(MMD の上限 {np.sqrt(2*kxy(0.0,var_g,var_g,sig_k)):.4f})")

$W_2$ は平均のずれにそのまま比例するのに対し、MMD は帯域幅 $0.5$ に対してずれが $2$ を超えると
ほぼ飽和する：ずれ $2.0$ で MMD $=1.1817$、ずれ $6.0$ でも $1.1922$（上限 $1.1922$）である。
帯域幅が小さいと**台が交わらない二分布について「離れている」ことしか分からなくなる**という
本文の指摘が数値で見える。一方 MMD は $O(n^2)$ の閉形式で計算でき標本複雑度が次元に依らない
（$W_1$ は $O(n^{-1/d})$）。用途で使い分ける。

最後に **Gromov--Wasserstein**（**節11.5.5**、式(11.24)、離散版は式(11.25)）。
$\boldsymbol{x}\in\mathbb{R}^{d_1}$ と $\boldsymbol{y}\in\mathbb{R}^{d_2}$ の
間に $\|\boldsymbol{x}-\boldsymbol{y}\|$ が書けない状況で、点そのものではなく**点対の距離**を比べる：

$$\mathrm{GW}_2^2=\min_{\boldsymbol{P}\in U(\boldsymbol{a},\boldsymbol{b})}\sum_{i,k}\sum_{j,l}
\bigl(C^{\mathcal X}_{ik}-C^{\mathcal Y}_{jl}\bigr)^2P_{ij}P_{kl}.$$

$\boldsymbol{P}$ について**二次**なので線形計画ではなく二次割当問題（NP 困難）である。
$n=6$ と小さいので**総当たり（$6!=720$ 通りの置換）**で厳密解を出し、
POT（`pot`）の反復解と突き合わせる。POT が入らない環境でも総当たりだけで結論は出る。

In [ ]:
from itertools import permutations

# POT は Colab の既定環境にないので、この節だけで導入する（マジックは使わない）
try:
    import ot
    HAS_OT = True
except ImportError:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pot"],
                       check=True, timeout=300)
        import ot
        HAS_OT = True
    except Exception:
        HAS_OT = False
print("POT:", "利用可" if HAS_OT else "利用不可（総当たりのみで進める）")


def dmat(Z):
    """列がサンプルの Z（d×n）から距離行列（n×n）を作る。"""
    return np.linalg.norm(Z[:, :, None] - Z[:, None, :], axis=0)


def gw2_brute(Ca, Cb):
    """等重み・同サイズのとき GW_2^2 を総当たりで厳密に解く。"""
    n = Ca.shape[0]
    return min((((Ca - Cb[np.ix_(p, p)]) ** 2).sum()) / n ** 2
               for p in permutations(range(n)))


rng = np.random.default_rng(3)
n_g = 6
Xg = rng.normal(size=(2, n_g))                                   # d×n = 2×6
ang = np.deg2rad(40.0)
R = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
Y_rot = R @ Xg + np.array([[3.0], [-2.0]])                       # 回転＋平行移動
Y_ref = np.diag([1.0, -1.0]) @ Xg                                # 鏡映
Y_rnd = rng.normal(size=(2, n_g)) * 2.5                          # 無関係な点群

Ca = dmat(Xg)
a_g = np.full(n_g, 1.0 / n_g)
rows = []
for nm, Yv in [(L("回転＋平行移動", "rotation+shift"), Y_rot),
               (L("鏡映", "reflection"), Y_ref),
               (L("無関係な点群", "unrelated"), Y_rnd)]:
    Cb = dmat(Yv)
    gw_b = gw2_brute(Ca, Cb)
    w2 = ot_lp(a_g, a_g, ((Xg[:, :, None] - Yv[:, None, :]) ** 2).sum(0))[0]
    gw_p = (float(ot.gromov.gromov_wasserstein2(Ca, Cb, a_g, a_g, "square_loss"))
            if HAS_OT else np.nan)
    rows.append((nm, gw_b, gw_p, w2))
    print(f"{nm:16s} GW_2^2(総当たり) = {gw_b:11.4e}   "
          f"GW_2^2(POT) = {gw_p:11.4e}   W_2^2 = {w2:8.4f}")

fig, axes = plt.subplots(1, 4, figsize=(14.5, 3.5))
for ax, (nm, Yv) in zip(axes[:3], [(L("回転＋平行移動", "rotation+shift"), Y_rot),
                                   (L("鏡映", "reflection"), Y_ref),
                                   (L("無関係な点群", "unrelated"), Y_rnd)]):
    Cb = dmat(Yv)
    best = min(permutations(range(n_g)),
               key=lambda p: ((Ca - Cb[np.ix_(p, p)]) ** 2).sum())
    ax.scatter(Xg[0], Xg[1], s=45, color=C["blue"], label=L("$\\mathcal{X}$", "X"))
    ax.scatter(Yv[0], Yv[1], s=45, color=C["red"], marker="s",
               label=L("$\\mathcal{Y}$", "Y"))
    for i, j in enumerate(best):
        ax.plot([Xg[0, i], Yv[0, j]], [Xg[1, i], Yv[1, j]], color=C["gray"], lw=0.8)
    ax.set_title(nm, fontsize=11); ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=9)
ax = axes[3]
xpos, floor = np.arange(3), 1e-3
ax.bar(xpos - 0.2, [max(r[1], floor) for r in rows], 0.4, color=C["green"],
       label=L("$\\mathrm{GW}_2^2$", "GW"))
ax.bar(xpos + 0.2, [r[3] for r in rows], 0.4, color=C["purple"],
       label=L("$W_2^2$", "W2"))
for i, r in enumerate(rows):                      # 0 は対数軸に描けないので値を書き添える
    ax.text(i - 0.2, floor * 1.3, "0" if r[1] < 1e-20 else f"{r[1]:.2f}",
            ha="center", fontsize=8, color=C["green"])
ax.set_ylim(floor, 100)
ax.set_yscale("log"); ax.set_xticks(xpos)
ax.set_xticklabels([L("回転", "rot"), L("鏡映", "refl"), L("無関係", "unrel")])
ax.set_title(L("GW と $W_2$ の比較", "GW vs W2")); ax.legend(fontsize=9)
plt.show()

等長なコピーに対して $\mathrm{GW}_2^2$ は**厳密に $0$**（総当たりで回転＋平行移動が $2.1\times10^{-31}$、
鏡映が $0$、いずれも機械精度）である一方、同じ二組で $W_2^2$ はそれぞれ
$15.99$、$3.34$ と座標のずれの分だけ増える。無関係な点群では
$\mathrm{GW}_2^2=1.081$ と大きい。図の灰色の線は最適な対応 $\boldsymbol{P}^\star$ を表す。

POT の反復解は等長なコピーでは総当たりと一致する（表示が $-10^{-15}$ 程度の負になるのは
丸め誤差で、$0$ と読む）が、無関係な点群では $1.812$ と総当たりの $1.081$ より大きい。
**目的関数が $\boldsymbol{P}$ について二次で非凸**なので、線形化と Sinkhorn を繰り返す反復は
局所最適に落ちうる。$n=6$ だから総当たりで確かめられたが、実問題では反復解が大域最適かどうかは
分からない——これは NP 困難な二次割当問題である。

**座標系を共有しない対象を比べられる**のが GW の利点だが、裏を返せば**向きや位置の情報は落ちる**
（鏡映を区別できない）。用途は「対応関係そのものを求めたい」場面——グラフの照合、
異モダリティの単一細胞データ（scRNA-seq と scATAC-seq）の統合、異言語の埋め込みの対応づけ——である。

## 11.6 単一細胞データ解析への応用（RECODE と scEGOT）

本文の**節11.6**に対応する。これまでの道具が一つの研究分野でどう組み合わさるかを見る。
**以下は論文の実装の再現ではなく、考え方を合成データで再現するデモである。**
実データも公開実装も使わず、本文が説明した骨格だけを数十行で書き下す。

### 11.6.1 RECODE 的な処理：次元削減をせずノイズだけ落とす

骨格は三段だった。(i) 遺伝子ごとのノイズ分散は観測から推定できる（ここでは既知とする）。
(ii) そのノイズ標準偏差でスケールするとノイズが等方になり、ノイズ由来の固有値が
Marchenko--Pastur の台にそろう。そこで**固有値を縮小**する。(iii) スケールを戻す。

比較するのは三者である：**素の観測値**、**PCA 打ち切り**（上位 $\hat k$ 成分をそのまま残す）、
**縮小**（同じ $\hat k$ 成分を $(\lambda_i-1)/\lambda_i$ 倍する）。
どれも $d\times n$ の行列を返すので、真の信号への距離が直接比較できる。

In [ ]:
rng = np.random.default_rng(0)
d_g, n_c, r_s = 200, 300, 12
sv = 60.0 * 0.75 ** np.arange(r_s)                     # 信号の特異値（減衰する）
U0 = np.linalg.qr(rng.normal(size=(d_g, r_s)))[0]
V0 = np.linalg.qr(rng.normal(size=(n_c, r_s)))[0]
Strue = (U0 * sv) @ V0.T                                # d×n 真の信号（列が細胞）
sd_gene = np.exp(rng.normal(0.0, 0.6, size=d_g))        # 遺伝子ごとの既知ノイズ SD
Xobs = Strue + sd_gene[:, None] * rng.normal(size=(d_g, n_c))

mu_g = Xobs.mean(axis=1, keepdims=True)
Z = (Xobs - mu_g) / sd_gene[:, None]                    # (ii) ノイズを等方にスケール
lam, U = np.linalg.eigh(Z @ Z.T / n_c)
lam, U = lam[::-1], U[:, ::-1]
mp_plus = (1.0 + np.sqrt(d_g / n_c)) ** 2               # MP 則の台の右端
khat = int((lam > mp_plus).sum())
A = U.T @ Z                                             # 各成分の係数（d×n）

fac_trunc = np.zeros(d_g); fac_trunc[:khat] = 1.0                       # PCA 打ち切り
fac_shrink = np.zeros(d_g)
fac_shrink[:khat] = np.maximum(lam[:khat] - 1.0, 0.0) / lam[:khat]      # 固有値の縮小


def back(fac):
    """スケールを戻して観測空間に返す（次元削減はしない：出力は d×n のまま）。"""
    return (U @ (fac[:, None] * A)) * sd_gene[:, None] + mu_g


rel = lambda Xh: np.linalg.norm(Xh - Strue) / np.linalg.norm(Strue)
print(f"MP の台の右端 = {mp_plus:.3f},  信号と判定された成分数 khat = {khat} (真のランク {r_s})")
print(f"素の観測値        : 相対誤差 {rel(Xobs):.4f}")
print(f"PCA 打ち切り(k={khat}) : 相対誤差 {rel(back(fac_trunc)):.4f}")
print(f"固有値の縮小      : 相対誤差 {rel(back(fac_shrink)):.4f}")
print(f"縮小の係数（上位5）: {np.round(fac_shrink[:5], 4)}")
g = int(np.argsort(sd_gene)[d_g // 2])                  # ノイズが中央値の遺伝子を一つ選ぶ
Xsh = back(fac_shrink)
print(f"遺伝子 {g}（ノイズ SD {sd_gene[g]:.2f}）の細胞ごとの RMSE : "
      f"素の観測値 {np.sqrt(np.mean((Xobs[g]-Strue[g])**2)):.3f} -> "
      f"縮小 {np.sqrt(np.mean((Xsh[g]-Strue[g])**2)):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.7))
ax = axes[0]
ax.semilogy(np.arange(1, 41), lam[:40], "o", ms=4, color=C["blue"])
ax.axhline(mp_plus, color=C["red"], ls="--", lw=1.2)
ax.text(20, mp_plus * 1.2, L("MP の台の右端", "MP bulk edge"), color=C["red"], fontsize=9)
ax.set_xlabel(L("固有値の番号", "index")); ax.set_ylabel(L("固有値", "eigenvalue"))
ax.set_title(L("スケール後の固有値", "eigenvalues after scaling"))
ax = axes[1]
names = [L("素の観測値", "raw"), L("PCA 打ち切り", "PCA truncation"), L("縮小", "shrinkage")]
vals = [rel(Xobs), rel(back(fac_trunc)), rel(back(fac_shrink))]
ax.bar(names, vals, color=[C["gray"], C["orange"], C["green"]])
for i, v in enumerate(vals):
    ax.text(i, v * 1.03, f"{v:.4f}", ha="center", fontsize=9)
ax.set_ylabel(L("真の信号への相対誤差", "relative error to truth"))
ax.set_title(L("三者の比較", "comparison"))
ax = axes[2]
ax.plot(Strue[g], Xobs[g], ".", ms=4, color=C["gray"], label=L("素の観測値", "raw"))
ax.plot(Strue[g], Xsh[g], ".", ms=4, color=C["green"], label=L("縮小", "shrinkage"))
lim = [Strue[g].min() - 1, Strue[g].max() + 1]
ax.plot(lim, lim, color=C["red"], lw=1)
ax.set_xlabel(L("真の値", "true")); ax.set_ylabel(L("推定値", "estimate"))
ax.set_title(L(f"遺伝子 {g} の細胞ごとの値", f"gene {g}, per cell")); ax.legend(fontsize=9)
plt.show()

スケール後の固有値は MP の台の右端 $3.300$ を境に、信号由来の $5$ 個だけが飛び出している
（真のランクは $12$ だが、下位の弱い成分はノイズに埋もれて検出できない）。
相対誤差は素の観測値 $3.693$ → PCA 打ち切り $0.733$ → 縮小 $0.674$ と下がる。
**打ち切った上位成分にもノイズが残っている**ので、$(\lambda_i-1)/\lambda_i$ 倍する分だけ縮小が勝つ。

重要なのは、この処理が**次元削減ではない**ことである。出力は $200\times300$ の遺伝子空間の行列の
ままで、上位 $\hat k$ 成分の座標に置き換えたわけではない。右図はノイズが中央値の遺伝子
（番号 $56$、ノイズ SD $0.96$）の細胞ごとの値で、細胞ごとの RMSE は
$0.880\to0.152$ に下がる。個々の遺伝子の値が補正されて返るので、
下流の解析（発現量の比較、希少集団の同定）がそのまま行える。
PCA が「捨てる」ことでノイズを除くのに対し、RECODE は「縮める」ことでノイズを除く、という
本文の対比がこの数値である。

### 11.6.2 scEGOT 的な処理：成分間の最適輸送で分化を追う

scRNA-seq は測定のために細胞を壊すので同じ細胞を追跡できない。時点 $t_1,t_2$ で別々の
細胞集団のスナップショットが得られるだけである。scEGOT は各時点を**混合ガウス分布**で表し、
細胞ではなく**成分のあいだ**でエントロピー正則化つき最適輸送を解く。
成分間コストにはガウス分布どうしの $W_2^2$ の閉形式
$\|\boldsymbol{m}_1-\boldsymbol{m}_2\|^2+\mathrm{tr}(\boldsymbol{\Sigma}_1+\boldsymbol{\Sigma}_2-2(\boldsymbol{\Sigma}_2^{1/2}\boldsymbol{\Sigma}_1\boldsymbol{\Sigma}_2^{1/2})^{1/2})$
を使う。得られた $P_{kl}$ を行で正規化すれば「細胞種 $k$ から細胞種 $l$ への移行率」が読める。

合成データは「1つの前駆集団が2つに分岐し、他の2集団はそのまま移動する」という筋書きにした。

In [ ]:
from scipy.linalg import sqrtm
from sklearn.mixture import GaussianMixture

rng = np.random.default_rng(1)
m1_true = np.array([[0.0, 0.0], [4.5, 1.5], [-4.5, 1.5]])
w1_true = np.array([0.5, 0.25, 0.25])
m2_true = np.array([[1.6, 4.5], [-1.6, 4.5], [6.5, 5.0], [-6.5, 5.0]])
w2_true = np.array([0.28, 0.22, 0.25, 0.25])
n1, n2 = 600, 800
lab1 = rng.choice(3, size=n1, p=w1_true)
lab2 = rng.choice(4, size=n2, p=w2_true)
X1 = (m1_true[lab1] + 0.7 * rng.normal(size=(n1, 2))).T      # d×n = 2×600（列が細胞）
X2 = (m2_true[lab2] + 0.6 * rng.normal(size=(n2, 2))).T      # d×n = 2×800

# sklearn は行がサンプルなので転置して渡す
g1 = GaussianMixture(3, covariance_type="full", random_state=0).fit(X1.T)
g2 = GaussianMixture(4, covariance_type="full", random_state=0).fit(X2.T)


def w2_gauss(mA, SA, mB, SB):
    """ガウス分布どうしの W_2^2（閉形式）。"""
    sB = np.real(sqrtm(SB))
    return float(np.sum((mA - mB) ** 2) + np.trace(SA + SB - 2 * np.real(sqrtm(sB @ SA @ sB))))


Cg = np.array([[w2_gauss(g1.means_[k], g1.covariances_[k], g2.means_[l], g2.covariances_[l])
                for l in range(4)] for k in range(3)])
eps_g = 0.05 * Cg.max()
Pg = sinkhorn_log(g1.weights_, g2.weights_, Cg, eps_g, n_iter=2000)
T = Pg / g1.weights_[:, None]                                 # 行で正規化＝移行率
print("成分間コスト行列 C（W_2^2）:\n", np.round(Cg, 2))
print("移行率 T = P / a（行和は 1）:\n", np.round(T, 3))
print("周辺制約の誤差:", f"{np.abs(Pg.sum(1)-g1.weights_).max():.2e}",
      f"{np.abs(Pg.sum(0)-g2.weights_).max():.2e}")
print(f"解いた輸送問題の大きさ: 成分間 {Cg.shape} = {Cg.size} 変数 / "
      f"細胞間なら ({n1}, {n2}) = {n1*n2} 変数")

# 重心射影による速度場：v(x) = sum_k gamma_k(x) sum_l T_kl m_l - x
resp = g1.predict_proba(X1.T)                                  # n1×3
tgt = resp @ (T @ g2.means_)                                   # n1×2
V = tgt.T - X1                                                 # d×n1

fig, axes = plt.subplots(1, 3, figsize=(14, 4.0))
ax = axes[0]
ax.scatter(X1[0], X1[1], s=6, color=C["blue"], alpha=0.45, label=L("時点 $t_1$", "time 1"))
ax.scatter(X2[0], X2[1], s=6, color=C["red"], alpha=0.45, label=L("時点 $t_2$", "time 2"))
ax.scatter(*g1.means_.T, s=110, marker="X", color=C["blue"], edgecolor="w", zorder=5)
ax.scatter(*g2.means_.T, s=110, marker="X", color=C["red"], edgecolor="w", zorder=5)
for k in range(3):
    for l in range(4):
        if T[k, l] > 0.05:
            ax.annotate("", xy=g2.means_[l], xytext=g1.means_[k],
                        arrowprops=dict(arrowstyle="->", lw=1 + 4 * T[k, l], color=C["gray"]))
ax.set_title(L("成分間の移行（矢印の太さ＝移行率）", "component-level transitions"))
ax.legend(fontsize=9, loc="upper center")
ax = axes[1]
sel = np.arange(0, X1.shape[1], 6)
ax.quiver(X1[0, sel], X1[1, sel], V[0, sel], V[1, sel], angles="xy",
          scale_units="xy", scale=1.0, width=0.004, color=C["green"])
ax.scatter(X2[0], X2[1], s=5, color=C["red"], alpha=0.25)
ax.set_title(L("重心射影による速度場", "velocity field (barycentric projection)"))
ax = axes[2]
im = ax.imshow(T, cmap="viridis", vmin=0, vmax=1); ax.grid(False)
for k in range(3):
    for l in range(4):
        ax.text(l, k, f"{T[k,l]:.2f}", ha="center", va="center",
                color="w" if T[k, l] < 0.6 else "k", fontsize=9)
ax.set_xlabel(L("時点 $t_2$ の成分", "components at $t_2$"))
ax.set_ylabel(L("時点 $t_1$ の成分", "components at $t_1$"))
ax.set_title(L("移行率行列 $T$", "transition matrix"))
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

移行率行列を見ると、$t_1$ の中央の成分が $t_2$ の二つの成分へ**分岐**し
（該当行の最大成分が $0.504$ と $0.429$）、両端の成分はそれぞれ一つの成分へ
ほぼ確定的に移る（$0.96$ 以上）。合成データの筋書きが輸送計画からそのまま読める。

**計算量の違い**が設計の要点である。細胞ごとに輸送を解くなら $n_1\times n_2=480{,}000$ 変数の
問題になるが、成分間なら $3\times4=12$ 変数で済む。時点数が増えても、成分数
（数個から数十個）しか効かない。しかも $P_{kl}$ が「細胞種 $k$ から $l$ への移行率」として
そのまま解釈できる。速度場は輸送計画からの**重心射影**
$\boldsymbol{x}\mapsto\int\boldsymbol{y}\,\mathrm{d}\pi(\boldsymbol{y}\mid\boldsymbol{x})$ を、混合ガウスの負担率
$\gamma_k(\boldsymbol{x})$ を通して細胞ごとに評価したものである。
エントロピー正則化が $\pi$ を滑らかにしているので安定に計算できる。

**この講義の道具がどこで使われたか**：RECODE は高次元統計（MP 則・固有値の縮小）、
scEGOT は最適輸送（Sinkhorn）と混合ガウス分布、両者の前処理に PCA。
異モダリティの統合には座標系を共有しないので 11.5 の Gromov--Wasserstein が候補になる。
繰り返すが、以上は**考え方のデモ**であって、実際の RECODE / scEGOT の実装は
ノイズ分散の推定・成分数の選択・複数時点の扱いなど、ここで省いた部分に本質がある。

## 演習

### 演習1（NTK の基本性質、講義ノートの**演習11.1**）

閉形式の NTK
$\Theta(\boldsymbol{x},\boldsymbol{x}')=k_\sigma(\boldsymbol{x},\boldsymbol{x}')+(\boldsymbol{x}^{\top}\boldsymbol{x}')(\pi-\vartheta)/(2\pi)$
について、(a) 正斉次性 $\Theta(c\boldsymbol{x},c\boldsymbol{x}')=c^2\Theta(\boldsymbol{x},\boldsymbol{x}')$（$c>0$）、
(b) 単位ベクトルが直交するとき $\Theta=1/(2\pi)$、$\boldsymbol{x}'=-\boldsymbol{x}$ のとき $\Theta=0$、
(c) Gram 行列が半正定値であること、を数値で確かめよ。

### 演習2（Isomap の $\boldsymbol{B}$ と近傍数、本文の警告ボックスの続き）

11.3 の Swiss roll で近傍数 $k$ を $4,6,8,12,20,40$ と変え、
$\boldsymbol{B}=-\frac12\boldsymbol{H}\boldsymbol{D}_G^{(2)}\boldsymbol{H}$ の負の固有値の絶対値の和が正の和に占める割合を求めよ。
$k$ を大きくすると近傍グラフが多様体からはみ出して測地距離の近似が悪くなる。
その影響がこの割合に現れるか。

### 演習3（核ノルム最小化が破綻する $2\times2$ の例、講義ノートの**演習11.4**）

$\boldsymbol{M}(x)=\begin{pmatrix}1&2\\3&x\end{pmatrix}$ の左上三成分だけが観測されている。
(a) $2\times2$ では $\|\boldsymbol{M}\|_*^2=\|\boldsymbol{M}\|_{\mathrm F}^2+2|\det\boldsymbol{M}|$ が成り立つことを
数値で確かめ、$\|\boldsymbol{M}(x)\|_*$ を $x$ の関数として描け。
(b) 核ノルムを最小にする $x$ と、ランクを $1$ にする $x$ を求め、両者が一致しないことを示せ。

In [ ]:
# 演習1：NTK の性質
# TODO: (a) c = 2.0 として Theta(cX, cX) と c^2 Theta(X, X) の最大差を求める
# TODO: (b) 直交する単位ベクトル、および x' = -x のときの Theta の値を出す
# TODO: (c) 乱数の 30 点で Gram 行列を作り、最小固有値が >= 0 か確かめる


# 演習2：近傍数 k と B の負の固有値
# TODO: k を [4, 6, 8, 12, 20, 40] と変え、kneighbors_graph → shortest_path → B と作り、
#       負の固有値の絶対値の和 / 正の和（%）を表にする（11.3 のコードを流用してよい）


# 演習3：2x2 の核ノルム
# TODO: (a) x のグリッドで ||M(x)||_* を SVD から計算し、
#       sqrt(||M||_F^2 + 2|det M|) と一致することを確かめて図にする
# TODO: (b) 最小点の x と、ランク 1 になる x を求めて比べる

## 演習の解答

In [ ]:
# ---- 演習1 ----
rng_e = np.random.default_rng(7)
Xe = rng_e.normal(size=(3, 5))                    # d×n
c = 2.0
print("(a) |Theta(cX,cX) - c^2 Theta(X,X)| の最大 =",
      f"{np.abs(ntk_relu(c*Xe, c*Xe) - c**2*ntk_relu(Xe, Xe)).max():.2e}")
e1 = np.array([[1.0], [0.0]]); e2 = np.array([[0.0], [1.0]])
print(f"(b) 直交: Theta = {ntk_relu(e1, e2)[0,0]:.6f}  (1/(2pi) = {1/(2*np.pi):.6f})")
print(f"    x' = -x : Theta = {ntk_relu(e1, -e1)[0,0]:.2e}")
Xe2 = rng_e.normal(size=(4, 30))
print(f"(c) Gram 行列の最小固有値 = {np.linalg.eigvalsh(ntk_relu(Xe2, Xe2)).min():.3e} (>= 0)")

# ---- 演習2 ----
print("\n(演習2) 近傍数 k と B の負の固有値")
ks_all, ks, ratios = [4, 6, 8, 12, 20, 40], [], []
for kk in ks_all:
    Dk = shortest_path(kneighbors_graph(X.T, kk, mode="distance"), method="D", directed=False)
    if not np.isfinite(Dk).all():          # 近傍グラフが非連結だと測地距離が inf になる
        print(f"  k = {kk:2d} : 近傍グラフが非連結（B を作れない）")
        continue
    ek = np.linalg.eigvalsh(-0.5 * H @ (Dk ** 2) @ H)
    rt = np.abs(ek[ek < 0]).sum() / ek[ek > 0].sum() * 100
    ks.append(kk); ratios.append(rt)
    print(f"  k = {kk:2d} : 負の固有値 {int((ek<0).sum()):3d} 個, |負|/正 = {rt:5.2f} %")

# ---- 演習3 ----
xs_e = np.linspace(-4, 12, 801)
nuc_svd = np.array([np.linalg.svd(np.array([[1.0, 2.0], [3.0, x]]),
                                  compute_uv=False).sum() for x in xs_e])
nuc_frm = np.sqrt(14 + xs_e ** 2 + 2 * np.abs(xs_e - 6))
print(f"\n(演習3a) SVD と ||M||_F^2+2|det M| の式の最大差 = "
      f"{np.abs(nuc_svd - nuc_frm).max():.2e}")
x_star = xs_e[int(np.argmin(nuc_svd))]
print(f"(演習3b) 核ノルム最小の x = {x_star:.3f} (核ノルム {nuc_svd.min():.4f}) / "
      f"ランク 1 にする x = 6 (核ノルム {np.sqrt(50):.4f})")

fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.8))
ax = axes[0]
ax.plot(ks, ratios, "o-", color=C["blue"])
ax.set_xlabel(L("近傍数 $k$", "number of neighbours k"))
ax.set_ylabel(L("|負の固有値| の和 / 正の和 (%)", "|neg| sum / pos sum (%)"))
ax.set_title(L("演習2：$k$ と $\\mathbf{B}$ の非半正定値性", "Ex.2: k vs indefiniteness of B"))
ax = axes[1]
ax.plot(xs_e, nuc_svd, color=C["blue"], lw=2, label=L("SVD から", "from SVD"))
ax.plot(xs_e[::20], nuc_frm[::20], "o", ms=4, color=C["orange"],
        label=L("$\\sqrt{\\|M\\|_F^2+2|\\det M|}$", "closed form"))
ax.axvline(x_star, color=C["green"], ls="--", lw=1.2, label=L("核ノルム最小", "min nuclear norm"))
ax.axvline(6.0, color=C["red"], ls="--", lw=1.2, label=L("ランク 1", "rank 1"))
ax.set_xlabel("$x$"); ax.set_ylabel(L("核ノルム", "nuclear norm"))
ax.set_title(L("演習3：$2\\times2$ の核ノルム", "Ex.3: nuclear norm"))
ax.legend(fontsize=8)
plt.show()

**演習1**：正斉次性は $\mathrm{ReLU}$ が1次正斉次であることの帰結で、実測の差は $0$。
直交する単位ベクトルでは $\Theta=1/(2\pi)=0.159155$、$\boldsymbol{x}'=-\boldsymbol{x}$ では $2\times10^{-17}$。
半正定値性は有限幅表示 $\Theta_m(\boldsymbol{x},\boldsymbol{x}')=\langle\psi(\boldsymbol{x}),\psi(\boldsymbol{x}')\rangle$
（$\psi=\nabla_{\boldsymbol{\theta}}f$）から従い、$m\to\infty$ の各点収束極限でも保たれる。実測の最小固有値は $1.06\times10^{-2}$。

**演習2**：$k=4$ では近傍グラフが**非連結**になり測地距離が $\infty$ を含むので $\boldsymbol{B}$ を作れない。
$k=6$ で $4.67\,\%$、$k=8$ で $4.69\,\%$ と小さいが、$k=12$ で $29.9\,\%$、
$k=20$ で $37.5\,\%$、$k=40$ で $33.2\,\%$ と一桁大きくなる。
$k$ を大きくすると近傍グラフが巻きの層をまたいで「近くに見えて遠い」点対を短絡し、
測地距離の近似が壊れるからである。$k$ には**連結性を保つ下限**と
**多様体からはみ出さない上限**があり、$\boldsymbol{B}$ の非半正定値性は後者の劣化を映す指標として読める。

**演習3**：$\|\boldsymbol{M}\|_*^2=\|\boldsymbol{M}\|_{\mathrm F}^2+2|\det\boldsymbol{M}|$ は
$\sigma_1^2+\sigma_2^2=\|\boldsymbol{M}\|_{\mathrm F}^2$ と $\sigma_1\sigma_2=|\det\boldsymbol{M}|$ から従う
（実測の最大差 $5.3\times10^{-15}$）。$x<6$ の枝では $14+x^2+2(6-x)=(x-1)^2+25$ なので最小は $x=1$、
核ノルム $5.0$。一方ランク $1$ にする $x=6$ では核ノルム $\sqrt{50}\approx7.07$ である。
観測数 $3$ は自由度 $r(2n-r)=3$ ちょうどで余裕がなく、コヒーレンスも高く
（真のランク1部分で $\mu_0=1.80$、上限 $2$ の $90\%$）、観測位置も一様ランダムでない。
**凸緩和は魔法ではない**。